In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [3]:
def load_and_clean_compas(path):
    """
    Carga del dataset Compas y limpieza base.
    """
    df = pd.read_csv(path)
    
    # Filtrado por calidad de datos
    df = df[
        (df['days_b_screening_arrest'] <= 30) & 
        (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') # 'O' es ordinario/menor
    ]
    
    # Selección de variables relevantes (Features) y Target
    features = [
        'sex', 'age', 'age_cat', 'race', 
        'juv_fel_count', 'juv_misd_count', 'juv_other_count', 
        'priors_count', 'c_charge_degree'
    ]
    target = 'two_year_recid'
    
    return df[features], df[target]

#### **Preprocesamiento**

In [4]:
X, y = load_and_clean_compas('./dataset/compas-scores-two-years.csv')

# Estrategia de estratificación (Target + Raza)
# Esto asegura que el sesgo racial se pueda medir con la misma precisión en train y test
stratify_cols = y.astype(str) + X['race'].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.30,        
    random_state=42, 
    stratify=stratify_cols # Estratificación multidimensional
)

# Identificación de tipos de columnas
numeric_features = ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']
categorical_features = ['sex', 'age_cat', 'race', 'c_charge_degree']

# Creación de Transformers
# Para numéricas: Imputación por mediana + Escalado estándar
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Para categóricas: Imputación por el más frecuente + OneHot (evitando multicolinealidad)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Ensamblaje del ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [ ]:
# Pipeline Final (Listo para añadir un estimador)
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

# Ejemplo de uso
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_processed = full_pipeline.fit_transform(X_train)